# FNO Rheology: PDMS / Sylgard

Physics-informed Fourier Neural Operator for predicting G'(ω) and G''(ω)
across crosslinker ratios. Single pipeline with:

- Optional gated spectral convolutions
- Optional physics loss (terminal slopes + Kramers-Kronig monotonicity)
- Crossover / plateau modulus / entanglement M_e extraction
- Interactive digital twin with experimental overlay and Excel export
- PDF report


## 1. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import torch
import torch.nn as nn
import torch.optim as optim

from scipy.interpolate import interp1d
from scipy.optimize import fsolve

# Colab-only imports (comment out if running locally)
from google.colab import drive, files
import ipywidgets as widgets
from IPython.display import display, clear_output

!pip install -q fpdf
from fpdf import FPDF


In [ ]:
# --- Paths ---
drive.mount('/content/drive')
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/FNO_Rheology_PDMS/Rheology_Data_PDMS_Sylgard.xlsx"
SAVE_DIR  = "/content/drive/MyDrive/Colab Notebooks/FNO_Rheology_PDMS/"
os.makedirs(SAVE_DIR, exist_ok=True)

# --- Model / training ---
MODES   = 11         # spectral modes retained (must be <= n_freq//2 + 1)
WIDTH   = 64
EPOCHS  = 1500
LR      = 1e-3
USE_GATE          = True     # gated spectral conv
USE_PHYSICS_LOSS  = True     # terminal slopes + KK monotonicity penalty

# --- Physics constants (for M_e from plateau modulus) ---
TEMP_K       = 298.15   # K
DENSITY_PDMS = 965      # kg/m^3
R_GAS        = 8.314    # J/(mol K)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 2. Data loading

Each sheet name is a crosslinker ratio; columns are `Omega`, `Gp`, `Gpp`.
All quantities are transformed to log10 before training.

In [ ]:
def prepare_data(file_path):
    """Return (inputs, targets, ratios).

    inputs  : (n_ratios, n_freq, 2)  -> [ratio, log10(omega)]
    targets : (n_ratios, n_freq, 2)  -> [log10(Gp), log10(Gpp)]
    ratios  : list of floats
    """
    xls = pd.ExcelFile(file_path)
    ins, targs, ratios = [], [], []
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet).sort_values('Omega').reset_index(drop=True)
        r = float(sheet)
        lf   = np.log10(df['Omega'].values)
        lgp  = np.log10(df['Gp'].values)
        lgpp = np.log10(df['Gpp'].values)
        ins.append(np.stack([np.full_like(lf, r), lf], axis=-1))
        targs.append(np.stack([lgp, lgpp], axis=-1))
        ratios.append(r)
    X = torch.tensor(np.array(ins),   dtype=torch.float32)
    Y = torch.tensor(np.array(targs), dtype=torch.float32)
    return X, Y, ratios


## 3. Model

`SpectralConv1d` with optional complex gating. `RheoFNO` is the shared
architecture used for both baseline and physics-informed runs; toggle
gating via `USE_GATE`.

In [ ]:
class SpectralConv1d(nn.Module):
    """1D spectral convolution keeping the first `modes` Fourier coefficients.

    If `gated=True`, the retained modes are modulated by a learned complex
    sigmoid gate, which suppresses noisy modes and focuses capacity on
    physical ones.
    """
    def __init__(self, in_channels, out_channels, modes, gated=False):
        super().__init__()
        self.modes = modes
        self.gated = gated
        scale = 1.0 / (in_channels * out_channels)
        self.weights = nn.Parameter(
            scale * torch.rand(in_channels, out_channels, modes, dtype=torch.cfloat)
        )
        if gated:
            self.gate = nn.Parameter(
                scale * torch.rand(in_channels, out_channels, modes, dtype=torch.cfloat)
            )

    def forward(self, x):
        B = x.shape[0]
        x_ft = torch.fft.rfft(x)
        out_ft = torch.zeros(
            B, self.weights.shape[1], x.size(-1) // 2 + 1,
            dtype=torch.cfloat, device=x.device,
        )
        W = self.weights
        if self.gated:
            g_ri = torch.view_as_real(self.gate)
            g = torch.complex(torch.sigmoid(g_ri[..., 0]), torch.sigmoid(g_ri[..., 1]))
            W = W * g
        out_ft[:, :, :self.modes] = torch.einsum(
            "bix,iox->box", x_ft[:, :, :self.modes], W
        )
        return torch.fft.irfft(out_ft, n=x.size(-1))


class RheoFNO(nn.Module):
    def __init__(self, modes=MODES, width=WIDTH, gated=USE_GATE):
        super().__init__()
        self.fc0   = nn.Linear(2, width)
        self.conv0 = SpectralConv1d(width, width, modes, gated=gated)
        self.conv1 = SpectralConv1d(width, width, modes, gated=gated)
        self.w0    = nn.Conv1d(width, width, 1)
        self.w1    = nn.Conv1d(width, width, 1)
        self.fc1   = nn.Linear(width, 128)
        self.fc2   = nn.Linear(128, 2)

    def forward(self, x):
        # x: (B, n_freq, 2)
        x = self.fc0(x).permute(0, 2, 1)          # (B, width, n_freq)
        x = torch.relu(self.conv0(x) + self.w0(x))
        x = torch.relu(self.conv1(x) + self.w1(x))
        x = x.permute(0, 2, 1)                    # (B, n_freq, width)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)                        # (B, n_freq, 2)


## 4. Physics-informed loss

Adds two soft penalties on top of MSE:

- **Terminal slopes**: in the low-frequency (terminal) region, viscoelastic
  liquids satisfy G' ~ ω² (slope 2 in log-log) and G'' ~ ω (slope 1).
- **Kramers-Kronig monotonicity**: G'(ω) should be non-decreasing in ω.

Set `USE_PHYSICS_LOSS = False` in the config cell for plain MSE.

In [ ]:
class RheoPhysicsLoss(nn.Module):
    def __init__(self, slope_weight=0.3, kk_weight=0.1, terminal_frac=0.15):
        super().__init__()
        self.mse = nn.MSELoss()
        self.sw = slope_weight
        self.kw = kk_weight
        self.terminal_frac = terminal_frac

    def forward(self, pred, target, log_freqs):
        loss = self.mse(pred, target)

        n_t = max(3, int(self.terminal_frac * pred.shape[1]))
        df  = log_freqs[1] - log_freqs[0]

        gp_slope  = (pred[:, 1:n_t, 0] - pred[:, :n_t - 1, 0]) / df
        gpp_slope = (pred[:, 1:n_t, 1] - pred[:, :n_t - 1, 1]) / df

        slope_err = ((gp_slope  - 2.0) ** 2).mean() + \
                    ((gpp_slope - 1.0) ** 2).mean()

        # KK-inspired monotonicity of G'
        kk_err = torch.relu(-(pred[:, 1:, 0] - pred[:, :-1, 0])).mean()

        return loss + self.sw * slope_err + self.kw * kk_err


## 5. Training

In [ ]:
def train_model(model, inputs, targets, log_freqs,
                epochs=EPOCHS, lr=LR, use_physics=USE_PHYSICS_LOSS,
                log_every=200):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = RheoPhysicsLoss() if use_physics else nn.MSELoss()

    history = []
    for epoch in range(epochs + 1):
        model.train()
        optimizer.zero_grad()
        pred = model(inputs)
        loss = criterion(pred, targets, log_freqs) if use_physics \
               else criterion(pred, targets)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
        if epoch % log_every == 0:
            print(f"Epoch {epoch:5d} | Loss {loss.item():.6f}")
    return history


## 6. Crossover and physics extraction

For each ratio, extract:
- **Crossover frequency** ω_c: where G'(ω) = G''(ω) (interpolated root)
- **Plateau modulus** G_N⁰: approximated by G'(ω_max)
- **Entanglement M_e**: (ρ R T) / G_N⁰


In [ ]:
def get_rheo_physics(model, inputs, ratios):
    model.eval()
    with torch.no_grad():
        preds = model(inputs).cpu().numpy()
    log_freqs = inputs[0, :, 1].cpu().numpy()

    rows = []
    for i, r in enumerate(ratios):
        try:
            f_diff = interp1d(
                log_freqs, preds[i, :, 0] - preds[i, :, 1],
                kind='cubic', fill_value="extrapolate",
            )
            root = fsolve(f_diff, x0=log_freqs.max())[0]
            omega_c = 10.0 ** root
            g_at_c  = 10.0 ** interp1d(log_freqs, preds[i, :, 0])(root)
        except Exception:
            omega_c, g_at_c = np.nan, np.nan

        gn0 = 10.0 ** preds[i, -1, 0]
        me  = (DENSITY_PDMS * R_GAS * TEMP_K) / gn0

        rows.append({
            'Ratio': r,
            'Crossover_Freq_rad_s':   omega_c,
            'Crossover_Modulus_Pa':   g_at_c,
            'Plateau_Modulus_Pa':     gn0,
            'Me_g_mol':               me,
        })
    return pd.DataFrame(rows), preds


## 7. Visualization

In [ ]:
def run_visuals(model, inputs, targets, ratios, save_dir=SAVE_DIR):
    model.eval()
    with torch.no_grad():
        preds = model(inputs).cpu().numpy()
    freqs = 10.0 ** inputs[0, :, 1].cpu().numpy()
    tgt_np = targets.cpu().numpy()

    os.makedirs(save_dir, exist_ok=True)

    # --- Fan plot: G' and G'' vs omega across ratios ---
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    cmap = plt.cm.viridis(np.linspace(0, 1, len(ratios)))
    for i, r in enumerate(ratios):
        ax[0].loglog(freqs, 10 ** tgt_np[i, :, 0], 'o', color=cmap[i], alpha=0.3)
        ax[0].loglog(freqs, 10 ** preds[i,  :, 0], '-', color=cmap[i], label=f'r={r}')
        ax[1].loglog(freqs, 10 ** tgt_np[i, :, 1], 'o', color=cmap[i], alpha=0.3)
        ax[1].loglog(freqs, 10 ** preds[i,  :, 1], '--', color=cmap[i])
    ax[0].set(title="Storage modulus G'",  xlabel="ω (rad/s)", ylabel="G' (Pa)")
    ax[1].set(title="Loss modulus G''",    xlabel="ω (rad/s)", ylabel="G'' (Pa)")
    ax[0].legend(bbox_to_anchor=(1.05, 1), fontsize=8)
    plt.tight_layout()
    fan_path = os.path.join(save_dir, "pdms_sylgard_fan_plot.png")
    plt.savefig(fan_path, dpi=150, bbox_inches='tight')
    plt.show()

    # --- Tan delta sensitivity heatmap (ratio, omega) -> tan delta ---
    n_grid = 50
    r_range = np.linspace(min(ratios), max(ratios), n_grid)
    l_freqs = np.log10(np.logspace(np.log10(freqs.min()),
                                   np.log10(freqs.max()), n_grid))
    td_grid = np.zeros((n_grid, n_grid))
    for i, ri in enumerate(r_range):
        t_in = torch.tensor(
            np.stack([np.full_like(l_freqs, ri), l_freqs], axis=-1),
            dtype=torch.float32,
        ).unsqueeze(0).to(inputs.device)
        with torch.no_grad():
            p = model(t_in).squeeze(0).cpu().numpy()
        td_grid[:, i] = 10 ** (p[:, 1] - p[:, 0])

    plt.figure(figsize=(10, 7))
    plt.pcolormesh(r_range, 10 ** l_freqs, td_grid,
                   norm=mcolors.LogNorm(0.1, 10),
                   cmap='RdYlBu_r', shading='gouraud')
    plt.yscale('log')
    plt.xlabel("Crosslinker ratio")
    plt.ylabel("ω (rad/s)")
    plt.colorbar(label='tan δ')
    plt.title("PDMS network sensitivity map")
    heat_path = os.path.join(save_dir, "pdms_sylgard_heatmap.png")
    plt.savefig(heat_path, dpi=150, bbox_inches='tight')
    plt.show()

    return fan_path, heat_path


def plot_physics_summary(phys_df):
    """Two-panel bar chart: crossover frequency and modulus vs ratio."""
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].bar(phys_df['Ratio'].astype(str), phys_df['Crossover_Freq_rad_s'], color='skyblue')
    ax[0].set(xlabel='Ratio', ylabel='ω_c (rad/s)', title='Crossover frequency')
    ax[0].grid(axis='y', linestyle='--', alpha=0.6)

    ax[1].bar(phys_df['Ratio'].astype(str), phys_df['Crossover_Modulus_Pa'], color='lightcoral')
    ax[1].set(xlabel='Ratio', ylabel='G_c (Pa)', title='Crossover modulus')
    ax[1].grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


## 8. Run: load, train, analyze

In [ ]:
inputs, targets, ratios = prepare_data(FILE_PATH)
inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
log_freqs = inputs[0, :, 1]
print(f"Loaded {len(ratios)} ratios, {inputs.shape[1]} frequency points each")

model = RheoFNO(modes=MODES, width=WIDTH, gated=USE_GATE).to(DEVICE)
history = train_model(model, inputs, targets, log_freqs)


In [ ]:
phys_df, _ = get_rheo_physics(model, inputs, ratios)
display(phys_df)
plot_physics_summary(phys_df)

fan_path, heat_path = run_visuals(model, inputs, targets, ratios, SAVE_DIR)


## 9. Interactive digital twin

Slider selects the crosslinker ratio. Toggle overlays the nearest
experimental curves. Export saves the current predicted curve as
`.xlsx`.

In [ ]:
def predict_curve(model, ratio, n_points=100, freq_decades=(-2, 3)):
    """Return (f, Gp, Gpp) on a log-spaced frequency grid."""
    model.eval()
    log_grid = np.linspace(freq_decades[0], freq_decades[1], n_points)
    t_in = torch.tensor(
        np.stack([np.full_like(log_grid, ratio), log_grid], axis=-1),
        dtype=torch.float32,
    ).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = model(t_in).squeeze(0).cpu().numpy()
    return 10 ** log_grid, 10 ** p[:, 0], 10 ** p[:, 1]


def _plot_prediction(ratio, show_exp=True):
    f, gp, gpp = predict_curve(model, ratio)
    gn0 = gp[-1]
    me  = (DENSITY_PDMS * R_GAS * TEMP_K) / gn0

    plt.figure(figsize=(11, 6))
    plt.loglog(f, gp,  'b-', lw=2.5, label=f"FNO G'  (r={ratio:.3f})")
    plt.loglog(f, gpp, 'r-', lw=2.5, label=f"FNO G'' (r={ratio:.3f})")
    plt.axhline(gn0, color='gray', ls=':', label=f"G_N⁰ ≈ {gn0:.2e} Pa")

    if show_exp:
        idx = int(np.abs(np.array(ratios) - ratio).argmin())
        r_near = ratios[idx]
        plt.loglog(10 ** inputs[idx, :, 1].cpu().numpy(),
                   10 ** targets[idx, :, 0].cpu().numpy(),
                   'bo', alpha=0.4, label=f"exp G'  (r={r_near})")
        plt.loglog(10 ** inputs[idx, :, 1].cpu().numpy(),
                   10 ** targets[idx, :, 1].cpu().numpy(),
                   'ro', alpha=0.4, label=f"exp G'' (r={r_near})")

    plt.title(f"PDMS digital twin | r={ratio:.3f} | M_e ≈ {me:.0f} g/mol")
    plt.xlabel("ω (rad/s)"); plt.ylabel("Modulus (Pa)")
    plt.grid(True, which='both', alpha=0.3)
    plt.legend(loc='lower right')
    plt.show()


# --- UI ---
_out = widgets.Output()
_slider = widgets.FloatSlider(
    value=float(np.mean(ratios)),
    min=min(ratios), max=max(ratios), step=0.001,
    description='Ratio:', layout=widgets.Layout(width='550px'),
    style={'description_width': 'initial'}, continuous_update=False,
)
_overlay = widgets.Checkbox(value=True, description='Overlay nearest exp')
_export  = widgets.Button(description='Export curve to Excel',
                          button_style='success', icon='download')

def _refresh(_=None):
    with _out:
        clear_output(wait=True)
        _plot_prediction(_slider.value, show_exp=_overlay.value)

def _export_click(_):
    ratio = _slider.value
    f, gp, gpp = predict_curve(model, ratio)
    df_out = pd.DataFrame({
        'Omega_rad_s':        f,
        'Storage_Modulus_Pa': gp,
        'Loss_Modulus_Pa':    gpp,
        'Tan_Delta':          gpp / gp,
    })
    fname = f"FNO_Prediction_Ratio_{ratio:.3f}.xlsx"
    df_out.to_excel(fname, index=False)
    files.download(fname)
    print(f"Exported: {fname}")

_slider.observe(_refresh, names='value')
_overlay.observe(_refresh, names='value')
_export.on_click(_export_click)

display(widgets.VBox([widgets.HBox([_slider, _overlay]), _export, _out]))
_refresh()


## 10. PDF report

In [ ]:
class PDFReport(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 15)
        self.cell(0, 10, 'PDMS FNO Rheology and Physics Report', 0, 1, 'C')

    def add_image_section(self, img_path, title, caption):
        self.set_font('Arial', 'B', 12); self.cell(0, 10, title, 0, 1)
        self.set_font('Arial', '', 10);  self.multi_cell(0, 5, caption)
        self.image(img_path, x=15, w=175); self.ln(5)

    def add_table(self, df, col_width=38, row_height=6):
        self.set_font('Arial', 'B', 9)
        for c in df.columns:
            self.cell(col_width, row_height + 1, str(c), 1)
        self.ln()
        self.set_font('Arial', '', 9)
        for _, row in df.iterrows():
            for val in row:
                txt = f"{val:.3e}" if isinstance(val, float) else str(val)
                self.cell(col_width, row_height, txt, 1)
            self.ln()


def build_pdf_report(phys_df, fan_path, heat_path, out_path):
    pdf = PDFReport()

    pdf.add_page()
    pdf.add_image_section(
        fan_path, "1. Frequency sweeps",
        "Experimental data (circles) vs FNO predictions (curves) across "
        "crosslinker ratios.",
    )

    pdf.add_page()
    pdf.add_image_section(
        heat_path, "2. Network sensitivity map",
        "tan δ as a function of ratio and ω. Contours near tan δ = 1 "
        "trace the liquid-to-solid gelation boundary.",
    )

    pdf.add_page()
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 10, "3. Derived physical properties", 0, 1)
    pdf.add_table(phys_df)

    pdf.output(out_path)
    print(f"Wrote {out_path}")


report_path = os.path.join(SAVE_DIR, "PDMS_Sylgard_Report.pdf")
build_pdf_report(phys_df, fan_path, heat_path, report_path)


## 11. Save model and results

In [ ]:
phys_df.to_csv(os.path.join(SAVE_DIR, "pdms_physics_results.csv"), index=False)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "pdms_fno_weights.pth"))
print(f"Saved weights and results to {SAVE_DIR}")
